# Ноутбук 1 — Загрузка данных CEAP-360VR и контроль качества

**Назначение.** Демонстрация процедуры загрузки исходного датасета CEAP-360VR
и формирования отчёта о качестве сигналов: пропуски, валидность айтрекинга,
длина записей, межсубъектные различия.

**Входы.** JSON-файлы датасета в `CEAP-360VR-Dataset-master/CEAP-360VR/`
(подкаталоги `2_QuestionnaireData`, `3_AnnotationData/Frame`,
`4_BehaviorData/Frame`, `5_PhysioData/Frame`).

**Выходы.** Сводная таблица качества по 32 участникам.

In [1]:
import sys
from pathlib import Path

# Корень notebooks/ должен быть в sys.path, чтобы импорты `from modules...` работали
NB_ROOT = Path.cwd()
if str(NB_ROOT) not in sys.path:
    sys.path.insert(0, str(NB_ROOT))

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from modules import config

print(f"Notebooks root: {NB_ROOT}")
print(f"Датасет: {config.DATASET_ROOT}")
print(f"Результаты: {config.RESULTS_DIR}")

Notebooks root: D:\Programming\Python\Diplom\notebooks
Датасет: D:\Programming\Python\Diplom\CEAP-360VR-Dataset-master\CEAP-360VR
Результаты: D:\Programming\Python\Diplom\source\results


## 1.1 Импорт модулей загрузчика

In [2]:
from modules import loader

## 1.2 Загрузка одного участника (демонстрация)

Загрузим данные участника `P1`, чтобы проверить структуру JSON и форму
получаемых DataFrame для каждой модальности.

In [3]:
pid = "P1"
ssq = loader.load_ssq_scores(pid)
print("SSQ pre/mid/post:")
print(ssq.to_string(index=False))

SSQ pre/mid/post:
participant phase  Nausea  Oculomotor  Disorientation  TotalScore
         P1   pre   28.62       15.16           27.84       26.18
         P1   mid   19.08       15.16            0.00       14.96
         P1  post    9.54        7.58            0.00        7.48


In [4]:
beh = loader.load_behavior(pid)
print(f"Поведенческие данные {pid}: {beh.shape}")
print(beh.head())

Поведенческие данные P1: (14100, 13)
  participant video  t_sec  head_pitch  head_yaw  gaze_pitch  gaze_yaw  \
0          P1    V1   0.00       1.180     1.250       0.587    -0.041   
1          P1    V1   0.04       1.172     1.239       0.778    -0.035   
2          P1    V1   0.08       1.158     1.167       0.810    -0.089   
3          P1    V1   0.12       1.130     1.092       0.778     0.098   
4          P1    V1   0.16       1.117     1.063       0.761     0.134   

   left_eye_pitch  left_eye_yaw  right_eye_pitch  right_eye_yaw  left_pupil  \
0           1.133        -0.417            0.040          0.335       5.580   
1           1.492        -0.399            0.065          0.329       5.581   
2           1.506        -0.428            0.113          0.250       5.570   
3           1.460        -0.169            0.097          0.365       5.566   
4           1.365        -0.062            0.156          0.330       5.558   

   right_pupil  
0        5.580  
1        

In [5]:
phy = loader.load_physio(pid)
print(f"Физиологические данные {pid}: {phy.shape}")
print(phy.head())

Физиологические данные P1: (14100, 10)
  participant video  t_sec       eda       bvp         hr       skt  \
0          P1    V1   0.00  0.165399  0.193771  85.530000  0.000000   
1          P1    V1   0.04  0.179468  0.118313  85.526798  0.000375   
2          P1    V1   0.08  0.193538  0.210542  85.523596  0.000750   
3          P1    V1   0.12  0.207608  0.458494  85.520394  0.001125   
4          P1    V1   0.16  0.221677  0.709096  85.517191  0.001500   

       acc_x  acc_y      acc_z  
0 -52.000000   23.0  30.000000  
1 -52.291183   23.0  30.000000  
2 -52.435832   23.0  30.000000  
3 -52.000000   23.0  29.158708  
4 -52.000000   23.0  30.000000  


In [6]:
ann = loader.load_annotations(pid)
print(f"Аннотации valence/arousal {pid}: {ann.shape}")
print(ann.head())

Аннотации valence/arousal P1: (14100, 5)


  participant video  t_sec  valence  arousal
0          P1    V1   0.00      5.0      5.0
1          P1    V1   0.04      5.0      5.0
2          P1    V1   0.08      5.0      5.0
3          P1    V1   0.12      5.0      5.0
4          P1    V1   0.16      5.0      5.0


## 1.3 Отчёт о качестве по всем 32 участникам

Запустим `loader.analyze_participant` для каждого участника — функция собирает
сводку: возраст, гендер, число окон, доля невалидных сэмплов pupil/gaze,
число IBI-интервалов, баллы SSQ во всех трёх замерах.

In [7]:
rows = []
for pid_i in config.PARTICIPANT_IDS:
    try:
        rows.append(loader.analyze_participant(pid_i))
    except Exception as e:
        print(f"  {pid_i}: пропущен — {e}")
report = pd.DataFrame(rows)
print(f"Участников: {len(report)}")
report.head(10)

Участников: 32


,participant,age,gender,vr_experience,n_videos_behavior,n_samples_behavior,invalid_left_pupil,invalid_right_pupil,nan_gaze,n_samples_physio,nan_eda,nan_hr,n_ibi,ssq_pre_total,ssq_mid_total,ssq_post_total,ssq_delta_post_pre
0,P1,22,Male,5-20 times,8,14100,0.0,0.0,0.0,14100,0.0,0.0,385,26.18,14.96,7.48,-18.70
1,P2,22,Male,Less than 5 times,8,14100,0.0,0.0,0.0,14100,0.0,0.0,59,14.96,-7.48,0.00,-14.96
2,P3,25,Female,Less than 5 times,8,14100,0.0,0.0,0.0,14100,0.0,0.0,306,41.14,41.14,52.36,11.22
3,P4,25,Female,Less than 5 times,8,14100,0.0,0.0,0.0,14100,0.0,0.0,401,7.48,11.22,11.22,3.74
4,P5,21,Female,Less than 5 times,8,14100,0.0,0.0,0.0,14100,0.0,0.0,580,48.62,37.40,33.66,-14.96
5,P6,18,Female,Less than 5 times,8,14100,0.0,0.0,0.0,14100,0.0,0.0,447,3.74,3.74,0.00,-3.74
6,P7,19,Male,Less than 5 times,8,14100,0.0,0.0,0.0,14100,0.0,0.0,599,0.00,7.48,0.00,0.00
7,P8,24,Male,More than 20 times,8,14100,0.0,0.0,0.0,14100,0.0,0.0,453,3.74,29.92,29.92,26.18
8,P9,26,Male,First time,8,14100,0.0,0.0,0.0,14100,0.0,0.0,413,0.00,33.66,67.32,67.32
9,P10,22,Female,Less than 5 times,8,14100,0.0,0.0,0.0,14100,0.0,0.0,479,3.74,22.44,22.44,18.70


## 1.4 Сводная статистика качества

In [8]:
print("Доля невалидных сэмплов левого зрачка:")
print(f"  среднее = {report['invalid_left_pupil'].mean():.3f}")
print(f"  максимум = {report['invalid_left_pupil'].max():.3f}")

print("\nДоля NaN в EDA:")
print(f"  среднее = {report['nan_eda'].mean():.3f}")
print(f"  максимум = {report['nan_eda'].max():.3f}")

print("\nSSQ post Total:")
print(f"  среднее = {report['ssq_post_total'].mean():.2f} ± "
      f"{report['ssq_post_total'].std():.2f}")
print(f"  медиана = {report['ssq_post_total'].median():.2f}")
print(f"  SSQ > 15 у {(report['ssq_post_total'] > 15).sum()}/32 участников")

Доля невалидных сэмплов левого зрачка:
  среднее = 0.000
  максимум = 0.000

Доля NaN в EDA:
  среднее = 0.000
  максимум = 0.000

SSQ post Total:
  среднее = 23.14 ± 25.85
  медиана = 13.09
  SSQ > 15 у 14/32 участников


## 1.5 Сохранение CSV-отчётов

In [9]:
summary_path = config.RESULTS_DIR / "participants_summary.csv"
ssq_path = config.RESULTS_DIR / "ssq_all.csv"
report.to_csv(summary_path, index=False)
ssq_all = pd.concat([loader.load_ssq_scores(p) for p in config.PARTICIPANT_IDS], ignore_index=True)
ssq_all.to_csv(ssq_path, index=False)
print(f"Сохранено: {summary_path}")
print(f"Сохранено: {ssq_path}")

Сохранено: D:\Programming\Python\Diplom\source\results\participants_summary.csv
Сохранено: D:\Programming\Python\Diplom\source\results\ssq_all.csv


## 1.6 Выводы

* Все 32 участника имеют валидные данные по всем модальностям (айтрекинг,
  физиология Empatica E4, аннотации valence/arousal, опросники).
* Фактическая частота Frame-данных составляет 25 Гц (а не 30 Гц, как заявлено
  в описании датасета), что верифицировано по шагу временной метки.
* Доля невалидных сэмплов айтрекинга в среднем ниже 10%, что укладывается
  в рабочий порог 30% для отбраковки окон.

Дальнейший шаг — построение признаковой матрицы из 2-секундных окон (ноутбук 2).